# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import ollama

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")  

In [11]:
openai = OpenAI()

In [ ]:


system_prompt = """
    You are a helpful technical tutor who answers questions about coding in 
    all languages, software engineering, data science and LLMs
    """
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

def chat(message, model, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # Include this turn's user message (Gradio does not add it to history before the handler runs).
    updated = history + [{"role": "user", "content": message}]
    messages = [{"role": "system", "content": system_prompt}] + updated

    if model == 'GPT':
        use_model = MODEL_GPT
        response = openai.chat.completions.create(
            model=use_model,
            messages=messages,
            stream=True,
        )
        reply = ""
        for chunk in response:
            if not chunk.choices:
                continue
            delta = chunk.choices[0].delta
            if delta and delta.content:
                reply += delta.content
                yield updated + [{"role": "assistant", "content": reply}]
    else:
        use_model = MODEL_LLAMA
        response = ollama.chat(
            model=use_model,
            messages=messages,
            stream=True,
        )
        reply = ""
        for chunk in response:
            # Ollama stream chunks are dicts, not OpenAI-style objects with .choices
            piece = (chunk.get("message") or {}).get("content") or ""
            if piece:
                reply += piece
                yield updated + [{"role": "assistant", "content": reply}]


In [ ]:
with gr.Blocks() as demo:
    with gr.Row():
        chatbot = gr.Chatbot(height=250,type="messages")
    with gr.Row():
        message = gr.Textbox(label="Chat with Tech AI assistant",interactive=True)
    with gr.Row():
        model_select = gr.Dropdown(choices=["GPT", "LLAMA"], value="GPT", interactive=True)
    with gr.Row():
        clear = gr.ClearButton([chatbot, message])
    

    message.submit(chat, inputs=[message, model_select, chatbot], outputs=chatbot)

demo.launch(inbrowser=True)
